In [1]:
import pandas as pd
import re

# Data Loading

In this step, the dataset is loaded from a CSV file into a pandas DataFrame. This dataset contains job postings labeled as real or fraudulent.

In [2]:
# Load your dataset
df_raw = pd.read_csv('fake_job_postings.csv')

# Preview data
df_raw.head()

,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0


# Initial Data Exploration

We perform an initial inspection of the dataset to understand its structure, data types, and identify missing values.

Key checks:
- Data shape
- Column types
- Missing values distribution
- Target variable balance (fraudulent vs non-fraudulent)

In [3]:
df_raw.shape

(17880, 18)

In [4]:
# Check missing values
missing_values = df_raw.isnull().sum()

print("Missing values per column:")
print(missing_values)

Missing values per column:
job_id                     0
title                      0
location                 346
department             11547
salary_range           15012
company_profile         3308
description                1
requirements            2696
benefits                7212
telecommuting              0
has_company_logo           0
has_questions              0
employment_type         3471
required_experience     7050
required_education      8105
industry                4903
function                6455
fraudulent                 0
dtype: int64


In [5]:
df_raw['location'].value_counts().head(20)

location
GB, LND, London          718
US, NY, New York         658
US, CA, San Francisco    472
GR, I, Athens            464
US, ,                    339
US, TX, Houston          269
US, IL, Chicago          255
US, DC, Washington       251
DE, BE, Berlin           221
NZ, N, Auckland          218
US, CA, Los Angeles      185
GB, , London             179
US, TX, Austin           174
US, CA, San Diego        164
GB, ,                    138
US, GA, Atlanta          135
GB, LND,                 131
US, OR, Portland         131
CA, ON, Toronto          123
US, MA, Boston           117
Name: count, dtype: int64

In [6]:
df_raw['fraudulent'].value_counts().head(20)

fraudulent
0    17014
1      866
Name: count, dtype: int64

In [7]:
df_raw['country'] = df_raw['location'].str.split(',').str[0].str.strip()

df_raw['country'].value_counts().head(20)

country
US    10656
GB     2384
GR      940
CA      457
DE      383
NZ      333
IN      276
AU      214
PH      132
NL      127
BE      117
IE      114
SG       80
HK       77
PL       76
IL       72
EE       72
FR       70
ES       66
AE       54
Name: count, dtype: int64

In [8]:
fraud_df = df_raw[df_raw['fraudulent'] == 1]

fraud_by_country = fraud_df['country'].value_counts()

fraud_by_country.head(10)

country
US    730
AU     40
GB     23
MY     12
CA     12
QA      6
BH      5
IN      4
PL      3
TW      2
Name: count, dtype: int64

# Data Cleaning & Preparation

A copy of the raw dataset is created to preserve the original data. Irrelevant columns such as 'salary_range' and 'department' are removed as they are not essential for text-based classification.

In [9]:
# Drop rows with missing values
df_clean_desc = df_raw.dropna(subset=['description'])


print("Shape before cleaning:", df_raw.shape)
print("Shape after cleaning:", df_clean_desc.shape)
print(df_clean_desc.columns)

Shape before cleaning: (17880, 19)
Shape after cleaning: (17879, 19)
Index(['job_id', 'title', 'location', 'department', 'salary_range',
       'company_profile', 'description', 'requirements', 'benefits',
       'telecommuting', 'has_company_logo', 'has_questions', 'employment_type',
       'required_experience', 'required_education', 'industry', 'function',
       'fraudulent', 'country'],
      dtype='object')


In [10]:
drop_cols = ['salary_range', 'department']
df_clean = df_clean_desc.drop(columns=drop_cols)

print("Before:", df_clean_desc.shape)
print("After:", df_clean.shape)

Before: (17879, 19)
After: (17879, 17)


# Filtering US Job Postings

The dataset is filtered to include only job postings from the United States. This ensures consistency in language and reduces noise from international variations.

In [11]:
# Filter dataset to include only US job postings
df_us = df_clean[df_clean['country'] == 'US'].copy()

print(df_us.shape)
print(df_us.columns)

(10656, 17)
Index(['job_id', 'title', 'location', 'company_profile', 'description',
       'requirements', 'benefits', 'telecommuting', 'has_company_logo',
       'has_questions', 'employment_type', 'required_experience',
       'required_education', 'industry', 'function', 'fraudulent', 'country'],
      dtype='object')


# Handling Missing Values

Missing values in textual columns are filled with the string "missing". This ensures that no data is lost while preserving information about missing fields, which may be indicative of fraudulent postings.

In [12]:
# Check missing values
missing_values_us = df_us.isnull().sum()

print("Missing values per column:")
print(missing_values_us)

Missing values per column:
job_id                    0
title                     0
location                  0
company_profile        2076
description               0
requirements           1775
benefits               4683
telecommuting             0
has_company_logo          0
has_questions             0
employment_type        1762
required_experience    4284
required_education     4477
industry               2682
function               3904
fraudulent                0
country                   0
dtype: int64


In [13]:
df_us['fraudulent'].value_counts().head(20)

fraudulent
0    9926
1     730
Name: count, dtype: int64

In [14]:
# Replace missing values with 'missing' to retain information
text_cols = ['title', 'description', 'company_profile', 'requirements', 'benefits']

df_us[text_cols] = df_us[text_cols].fillna('missing')

cat_cols = ['employment_type','required_experience', 'required_education', 'industry', 'function']

df_us[cat_cols] = df_us[cat_cols].fillna('unknown')

In [15]:
# Check missing values
missing_values_us = df_us.isnull().sum()

print("Missing values per column:")
print(missing_values_us)

Missing values per column:
job_id                 0
title                  0
location               0
company_profile        0
description            0
requirements           0
benefits               0
telecommuting          0
has_company_logo       0
has_questions          0
employment_type        0
required_experience    0
required_education     0
industry               0
function               0
fraudulent             0
country                0
dtype: int64


# Feature Engineering

Additional features are created to enhance the model's ability to detect fraudulent job postings:

- Missing indicators: Flags for missing 'requirements', 'company_profile', and 'benefits'
- has_number: Detects presence of numbers in job descriptions (potential scam indicator)
- desc_length: Measures the length of job descriptions (fake jobs tend to be shorter)

In [16]:
df_us['missing_requirements'] = (df_us['requirements'] == "missing").astype(int)
df_us['missing_company_profile'] = (df_us['company_profile'] == "missing").astype(int)
df_us['missing_benefits'] = (df_us['benefits'] == "missing").astype(int)

df_us['has_number'] = df_us['description'].str.contains(r'\d').astype(int)

# Feature: length of job description (fake jobs tend to be shorter)
df_us['desc_length'] = df_us['description'].str.split().str.len()

df_us[['missing_requirements','missing_company_profile','missing_benefits']].mean()

missing_requirements       0.166573
missing_company_profile    0.194820
missing_benefits           0.439471
dtype: float64

# Text Preprocessing

Text cleaning is applied to normalize the data:
- Convert text to lowercase
- Remove newline characters
- Remove extra whitespace
- Remove URLs
- Remove HTML

Minimal preprocessing is used to preserve semantic meaning, which is important for transformer-based models like BERT.

In [17]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    return text.strip()

us_cols = [
    'title',
    'company_profile',
    'description',
    'requirements',
    'benefits',
    'industry',
    'function'
]

for col in us_cols:
    df_us[col] = df_us[col].apply(clean_text)

In [18]:
# Combine all relevant text fields into a single input for BERT
df_us['full_text'] = (
    "TITLE: " + df_us['title'] + " " +
    "DESC: " + df_us['description'] + " " +
    "REQ: " + df_us['requirements'] + " " +
    "COMPANY: " + df_us['company_profile'] + " " +
    "BENEFITS: " + df_us['benefits'] + " " +
    "INDUSTRY: " + df_us['industry'] + " " +
    "FUNCTION: " + df_us['function']
)

# Save Processed Dataset

The cleaned and processed dataset is saved for use in model training and evaluation.

In [19]:
# Save cleaned dataset
df_us.to_csv('job_scam_cleaned.csv', index=False)

# Tokenization

Tokenization is performed using the pre-trained tokenizer associated with the transformer model, ensuring compatibility with BERT and RoBERTa input formats

BERT Tokenization

In [20]:
from transformers import BertTokenizer

tokenizer_bert = BertTokenizer.from_pretrained('bert-base-uncased')

encoded_bert = tokenizer_bert(
    df_us['full_text'].tolist(),
    padding='max_length',
    truncation=True,
    max_length=256,
    return_tensors='pt'
)

BERT Validation

In [21]:
print(encoded_bert[0])

Encoding(num_tokens=256, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])


In [22]:
print(type(encoded_bert))

<class 'transformers.tokenization_utils_base.BatchEncoding'>


In [23]:
print("Keys:", encoded_bert.keys())

print("Input IDs shape:", encoded_bert['input_ids'].shape)
print("Attention mask shape:", encoded_bert['attention_mask'].shape)

labels = df_us['fraudulent'].values
print("Labels length:", len(labels))

assert encoded_bert['input_ids'].shape[0] == len(labels)

print("Sample input length:", len(encoded_bert['input_ids'][0]))
print("BERT Tokenization validation passed!")

Keys: KeysView({'input_ids': tensor([[ 101, 2516, 1024,  ..., 5939, 6873,  102],
        [ 101, 2516, 1024,  ..., 3208, 1521,  102],
        [ 101, 2516, 1024,  ..., 5656, 2005,  102],
        ...,
        [ 101, 2516, 1024,  ...,    0,    0,    0],
        [ 101, 2516, 1024,  ..., 1011, 2152,  102],
        [ 101, 2516, 1024,  ..., 2491, 2007,  102]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]])})
Input IDs shape: torch.Size([10656, 256])
Attention mask shape: torch.Size([10656, 256])
Labels length: 10656
Sample input length: 256
BERT Tokenization validation passe

RoBERTa Tokenization

In [24]:
from transformers import RobertaTokenizer

tokenizer_roberta = RobertaTokenizer.from_pretrained('roberta-base')

encoded_roberta = tokenizer_roberta(
    df_us['full_text'].tolist(),
    padding='max_length',
    truncation=True,
    max_length=256,
    return_tensors='pt'
)

RoBERTa Validation

In [25]:
print(encoded_roberta[0])

Encoding(num_tokens=256, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])


In [26]:
print(type(encoded_roberta))

<class 'transformers.tokenization_utils_base.BatchEncoding'>


In [27]:
print("Keys:", encoded_roberta.keys())

print("Input IDs shape:", encoded_roberta['input_ids'].shape)
print("Attention mask shape:", encoded_roberta['attention_mask'].shape)

labels = df_us['fraudulent'].values
print("Labels length:", len(labels))

assert encoded_roberta['input_ids'].shape[0] == len(labels)

print("Sample input length:", len(encoded_roberta['input_ids'][0]))
print("RoBERTa Tokenization validation passed!")

Keys: KeysView({'input_ids': tensor([[    0, 47217,  3850,  ...,     8,  3187,     2],
        [    0, 47217,  3850,  ..., 13759,     7,     2],
        [    0, 47217,  3850,  ...,  5731,    10,     2],
        ...,
        [    0, 47217,  3850,  ...,     1,     1,     1],
        [    0, 47217,  3850,  ...,   676,    19,     2],
        [    0, 47217,  3850,  ...,     6, 13803,     2]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]])})
Input IDs shape: torch.Size([10656, 256])
Attention mask shape: torch.Size([10656, 256])
Labels length: 10656
Sample input length: 256
RoBERTa Tokenization validation passed!
